# Backpropagation From Scratch - NumPy (Loop-Based)

This is the first of two notebooks implementing backpropagation for a 
3-layer neural network using only NumPy - no TensorFlow, no PyTorch, 
no autograd.

This version is deliberately written with explicit nested loops over 
each neuron and weight, instead of vectorized matrix operations. The 
goal here isn't efficiency - it's visibility. Writing `dJ_dW3[i][j] = 
a2[i] * dJ_dz3[j]` forces me to know exactly which weight connects 
which neuron to which, instead of trusting a matrix multiply to get 
it right.

**Architecture:** 2 inputs -> 3 neurons -> 2 neurons -> 1 output, 
sigmoid activation throughout.

**What this notebook covers:**
- Forward propagation through a dense layer, implemented manually
- The chain rule applied layer by layer to compute gradients 
  (backpropagation)
- A single step of gradient descent on one training example

**Next notebook:** `vectorized-backprop-numpy.ipynb` takes this same 
network and the same math, and rewrites it using matrix operations 
(`np.matmul`) instead of loops, then adds a real training loop over 
multiple epochs.

In [119]:
import numpy as np

In [23]:
def sigmoid(z): 
    return 1 / (1 + np.exp(-z))

In [45]:
def my_dense(a_in, W, b, g):
    """
    Args: 
     a_in(ndarray (n,)): 1 example
     W(ndarray (m,n)): Weights for each example
     b(ndarray (n)): bias vector, j units
     g: activation function (sigmoid, relu, etc.)

    Returns:
     a_out(ndarray (n,)): n units
    """
    units = W.shape[1]
    a_out = np.zeros(units)
    z = np.zeros(units)

    for n in range(units):
        w = W[:, n]
        z[n] = np.dot(w, a_in) + b[n]
        a_out[n] = g(z[n])
    return z, a_out

In [46]:
x_tst = 0.1*np.arange(1,3,1).reshape(2,)  # (1 examples, 2 features)
W_tst = 0.1*np.arange(1,7,1).reshape(2,3) # (2 input features, 3 output features)
b_tst = 0.1*np.arange(1,4,1).reshape(3,)  # (3 features)
A_tst = my_dense(x_tst, W_tst, b_tst, sigmoid)
print(A_tst)

(array([0.19, 0.32, 0.45]), array([0.54735762, 0.57932425, 0.61063923]))


In [54]:
def my_sequential(X, W1, b1, W2, b2, W3, b3):
    z1, a1 = my_dense(X, W1, b1, sigmoid)
    z2, a2 = my_dense(a1, W2, b2, sigmoid)
    z3, a3 = my_dense(a2, W3, b3, sigmoid)
    return z1, a1, z2, a2, z3, a3

In [115]:
X = np.array([1.0, 2.0])

# Layer 1: 2 inputs to 3 neurons
W1 = np.array([
    [0.1, 0.2, 0.3],
    [0.4, 0.5, 0.6]
])
b1 = np.array([0.1, 0.1, 0.1])

# Layer 2: 3 neurons to 2 neurons
W2 = np.array([
    [0.1, 0.2],
    [0.3, 0.4],
    [0.5, 0.6]
])
b2 = np.array([0.1, 0.1])

# Layer 3: 2 neurons to 1 neuron
W3 = np.array([
    [0.7],
    [0.8]
])
b3 = np.array([0.1])

z1, a1, z2, a2, z3, a3 = my_sequential(X, W1, b1, W2, b2, W3, b3)
print(a3)

[0.76509215]


Here we begin Backpropogation: 

In [117]:
y = np.array([1.0]) # Suppose y = 1.0

J = (1/2) * (y_hat - y)**2

Gradient Descent: 
w = w - alpha * dJ_dw
b = b - alpha * dJ_db 

In [118]:
# J = (1/2) * (a3 - y)**2
# To find: dJ_dz3 = dJ_da3 * da3_dz3
dJ_da3 = (a3 - y)
da3_dz3 = a3 * (1 - a3)
dJ_dz3 = dJ_da3 * da3_dz3

# dW3
dJ_dW3 = np.zeros_like(W3)

# dJ_dw1 = dJ_dz3 * dz3_dw1

for j in range(W3.shape[1]):
    for i in range(W3.shape[0]):
        dJ_dW3[i][j] = a2[i] * dJ_dz3[j]

dJ_db3 = dJ_dz3.copy()

In [92]:
dJ_da2 = np.zeros_like(a2)

for i in range(len(a2)):
    for j in range(len(dJ_dz3)):
        dJ_da2[i] += W3[i][j] * dJ_dz3[j] # dJ_da2 = dJ_dz3 * dz3_da2

da2_dz2 = a2 * (1 - a2)

dJ_dz2 = dJ_da2 * da2_dz2


# dW2
dJ_dW2 = np.zeros_like(W2)

for j in range(W2.shape[1]):
    for i in range(W2.shape[0]):
        dJ_dW2[i, j] = a1[i] * dJ_dz2[j]

dJ_db2 = dJ_dz2.copy()

In [93]:
dJ_da1 = np.zeros_like(a1)

for i in range(len(a1)):
    for j in range(len(dJ_dz2)):
        dJ_da1[i] += W2[i, j] * dJ_dz2[j]

da1_dz1 = a1 * (1 - a1)

dJ_dz1 = dJ_da1 * da1_dz1


# dW1
dJ_dW1 = np.zeros_like(W1)

for j in range(W1.shape[1]):
    for i in range(W1.shape[0]):
        dJ_dW1[i, j] = X[i] * dJ_dz1[j]

dJ_db1 = dJ_dz1.copy()

In [112]:
alpha = 0.1

In [113]:
# Gradient Descent
W1 = W1 - alpha * dJ_dW1
b1 = b1 - alpha * dJ_db1

W2 = W2 - alpha * dJ_dW2
b2 = b2 - alpha * dJ_db2

W3 = W3 - alpha * dJ_dW3
b3 = b3 - alpha * dJ_db3

In [114]:
z1, a1, z2, a2, z3, a3 = my_sequential(X, W1, b1, W2, b2, W3, b3)
print(a3)

[0.77344166]
